# Receive OpenLineage events from Airflow

Runs the *HarvestOpenLineageEvents::CreateAsCatalogTargetGovernanceActionProcess* governance action process. It catalogs the `openlineage.events` Kafka topic and makes it a catalog target of the **OpenLineageKafkaListener** integration connector, which then passes each event to the **OpenLineageCataloguer** to record as lineage.

Airflow sends its events to the OpenLineage proxy backend in shared-infra (port 6000), which publishes them to that topic. The host and port are those Egeria uses to reach Kafka: the shared Kafka server's external listener, `host.docker.internal:9194` (override with `OPENLINEAGE_KAFKA_HOST` / `OPENLINEAGE_KAFKA_PORT`).

The Portal's bootstrap monitor runs this notebook after a demo reset (see `_batch.json`). The step-by-step version is `workbooks/cataloguing-and-surveys/apache-kafka/kafka-open-lineage-events.ipynb`.

In [ ]:
import os

from pyegeria import AutomatedCuration

view_server = os.environ.get("EGERIA_VIEW_SERVER", "qs-view-server")
url = os.environ.get("EGERIA_PLATFORM_URL", "https://localhost:9443")
user_id = os.environ.get("EGERIA_USER", "erinoverview")
user_pwd = os.environ.get("EGERIA_USER_PASSWORD", "secret")

automated_curation = AutomatedCuration(view_server, url, user_id, user_pwd)
automated_curation.create_egeria_bearer_token(user_id, user_pwd)

request_parameters = {
    "serverName": "LocalKafka2",
    "hostIdentifier": os.environ.get("OPENLINEAGE_KAFKA_HOST", "host.docker.internal"),
    "portNumber": os.environ.get("OPENLINEAGE_KAFKA_PORT", "9194"),
    "description": "Receive open lineage events via Kafka Topic.",
    "topicDescription": "Open Lineage Events for Backend Proxy",
    "versionIdentifier": "V1.0",
    "fullTopicName": "openlineage.events",
    "shortTopicName": "openlineage.events",
    "eventDirection": "inOut",
}

process_guid = automated_curation.initiate_gov_action_process(
    "HarvestOpenLineageEvents::CreateAsCatalogTargetGovernanceActionProcess",
    None, None, None, request_parameters, None, None,
)
print(f"Started HarvestOpenLineageEvents: {process_guid}")